## Building from scratch a fully functional Text Generation LLM finetuned with a Q&A dataset(SQUAD)

### Project Exception Class

In [ ]:
class ProjectException(Exception):
    """this is a base exception class for the entire project"""
    def __init__(self, message, original_exception: Exception=None):
        super().__init__(message)
        self.message = message
        self.original_exception = original_exception
    def __str__(self):
        if self.original_exception:
            return f"{self.message}\n Caused by: {repr(self.original_exception)}"
        else:
            return self.message

### Data Download and Caching

In [ ]:
#you need to do this due to kagglehub authentication errors
#check if kagglehub sees your auth credentials
#if false, you need to create a legacy api token- it  will download a json file with your credentials
#uplaod that json file to your colab notebook and grant colab permission to your kaggle api key in secrets
import os

print(os.path.exists("/root/.kaggle/kaggle.json"))

In [ ]:
#next run the following
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
#now this will store your json file containing your creds into the kaggle cache file
!mkdir -p ~/.kaggle #mkdir to make kaggle dir in ~

In [ ]:
!cp kaggle.json ~/.kaggle/ #copy the kaggle.json to that directory

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json #change the file type to have certain privileges

In [ ]:
import kagglehub
import os

def download_datasets():
    """downloads the pre-training and Q&A dataset using kagglehub
    kagglehub automatically caches downloads in ~/.cache/kagglehub so repeated calls are indempotent and fast"""
    try:
        print("donwloading wikitext-103 (pre-training corpus)...")
        pretrain_path = kagglehub.dataset_download("vadimkurochkin/wikitext-103")
        print("downloading SQuAD v1.1 (Q&A dataset)...")
        squad_path = kagglehub.dataset_download("akashdesarda/squad-v11") #for simplicity but in actual sense i will use the squad2.0 (buildformacarov/squad-20)
        print("downloads complete (cached if already present)")
        return pretrain_path, squad_path
    except Exception as e:
        raise ProjectException("failed to download datasets", e)

# Usage
pretrain_dir, qa_dir = download_datasets()
print("Pre-training data directory:", pretrain_dir)
print("Q&A data directory:", qa_dir)

# List contents to understand the structure
print("\nContents of pre-training directory:")
print(os.listdir(pretrain_dir))
print("\nContents of Q&A directory:")
print(os.listdir(qa_dir))

### EDA for the datasets

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter

def eda_pretrain(data_dir):
    """performs EDA on the pre-training dataset"""
    train_file = None
    for fname in os.listdir(data_dir):
        if fname == "wiki.train.tokens": # Changed to look for the specific file name 'wiki.train.tokens'
            train_file = os.path.join(data_dir, fname)
            break
    if not train_file:
        raise ProjectException(f"No 'wiki.train.tokens' file found in the directory: {data_dir}.")

    with open(train_file, 'r',encoding="utf-8") as f:
        text = f.read()

    #basic counts
    num_lines = len(text)

    #tokenize each line by whitespace for length stats
    line_token_counts = [len(line.split()) for line in text.split('\n')]
    total_tokens = sum(line_token_counts)
    avg_tokens_per_line = total_tokens / num_lines if num_lines > 0 else 0
    max_tokens_per_line = max(line_token_counts)
    min_tokens_per_line = min(line_token_counts)

    #vocabulary size (case-sensitive, based on whitespace splitting)
    vocab = set()
    for line in text.split('\n'):
        vocab.update(line.split())
    vocab_size = len(vocab)

    #print
    print("=== Pretraining data (wikitext-103)===")
    print(f"Number of lines: {num_lines}")
    print(f"Total tokens: {total_tokens}")
    print(f"Average tokens per line: {avg_tokens_per_line}")
    print(f"Max tokens per line: {max_tokens_per_line}")
    print(f"Min tokens per line: {min_tokens_per_line}")
    print(f"Vocabulary size: {vocab_size}")
    print(f"samples lines:\n{text[1000:1500]}")
    print("\n")
    #plots
    plt.figure(figsize=(10, 6))
    sns.histplot(line_token_counts, bins=50, kde=True, color='skyblue')
    plt.title('Distribution of Line Token Counts')
    plt.xlabel('Token Count')
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
##do the same for the qa_dataset which is a csv
def eda_qa(data_dir):
    """performs EDA on the Q&A dataset"""
    train_file = None
    for fname in os.listdir(data_dir):
        # Prioritize SQuAD-v1.1.csv if available, otherwise take any .csv
        if fname == "SQuAD-v1.1.csv":
            train_file = os.path.join(data_dir, fname)
            break
        elif fname.endswith(".csv") and train_file is None: # only assign if not already found v1.1
            train_file = os.path.join(data_dir, fname)

    if not train_file:
        raise ProjectException("No training CSV file found in the directory.")

    df = pd.read_csv(train_file)

    # Check available columns in the DataFrame
    available_cols = df.columns.tolist()
    print(f"Available columns in QA dataset: {available_cols}")

    qa_data = []
    for index, row in df.iterrows():
        qa_entry = {
            "title": row["title"] if "title" in available_cols else None,
            "context": row["context"] if "context" in available_cols else None,
            "question": row["question"] if "question" in available_cols else None,
            "answer": row["answer"] if "answer" in available_cols else None,
        }
        if "answer_start" in available_cols:
            qa_entry["answer_start"] = row["answer_start"]
        if "answer_end" in available_cols:
            qa_entry["answer_end"] = row["answer_end"]
        qa_data.append(qa_entry)

    #store as a json in a writable directory (e.g., /content/)
    output_json_path = os.path.join("/content/", "qa_data.json")
    with open(output_json_path, "w") as f:
        json.dump(qa_data, f,indent=4)
    print(f"QA data saved to {output_json_path}")

    num_qa = len(qa_data)
    # Ensure context, question, answer are treated as strings before splitting
    context_lengths = [len(str(item["context"]).split()) for item in qa_data]
    question_lengths = [len(str(item["question"]).split()) for item in qa_data]
    answer_lengths = [len(str(item["answer"]).split()) for item in qa_data]

    #print
    print("=== Q&A data (SQUAD v1.1)===")
    print(f"Number of questions and answers: {num_qa}")
    print(f"Average context length: {np.mean(context_lengths)}")
    print(f"Average question length: {np.mean(question_lengths)}")
    print(f"Average answer length: {np.mean(answer_lengths)}")
    print(f"Max context length: {np.max(context_lengths)}")
    print(f"Max question length: {np.max(question_lengths)}")
    print(f"Max answer length: {np.max(answer_lengths)}")
    print(f"Min context length: {np.min(context_lengths)}")
    print(f"Min question length: {np.min(question_lengths)}")
    print(f"Min answer length: {np.min(answer_lengths)}")
    print(f"samples:\n")
    for i, qa in enumerate(qa_data[:5]):
        print(f"Question {i+1}: {qa['question']}")
        print(f"Context:first 200 chars {i+1}: {str(qa['context'])[:200]}")
        print(f"Answer {i+1}: {qa['answer']}")
        print("\n")

    #distributions
    fig,axes = plt.subplots(1,3,figsize=(15,5))
    axes[0].hist(context_lengths,bins=50,edgecolor="black")
    axes[0].set_title("Context Length Distribution")
    axes[0].set_xlabel("Length")
    axes[0].set_ylabel("Frequency")
    axes[1].hist(question_lengths,bins=50,edgecolor="black")
    axes[1].set_title("Question Length Distribution")
    axes[1].set_xlabel("Length")
    axes[1].set_ylabel("Frequency")
    axes[2].hist(answer_lengths,bins=50,edgecolor="black")
    axes[2].set_title("Answer Length Distribution")
    axes[2].set_xlabel("Length")
    axes[2].set_ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
# @title
import os
import json

try:
    # Always call download_datasets to get the fresh paths
    pretrain_dir_base, qa_dir = download_datasets()

    final_pretrain_path = None
    target_file = "wiki.train.tokens" # Changed target file to wiki.train.tokens

    # Try a few common nested patterns for Kaggle datasets
    possible_paths_to_check = [
        pretrain_dir_base,
        os.path.join(pretrain_dir_base, "wikitext-103"), # Common nested structure
        os.path.join(pretrain_dir_base, os.path.basename(pretrain_dir_base)) # Sometimes dataset name is repeated as subdir
    ]

    # Iterate through possible base paths and check for the target file
    checked_paths = set()
    for p_base in possible_paths_to_check:
        if not os.path.exists(p_base):
            continue
        if p_base in checked_paths:
            continue
        checked_paths.add(p_base)

        full_file_path = os.path.join(p_base, target_file)
        if os.path.isfile(full_file_path): # Using isfile for a more specific check
            final_pretrain_path = p_base
            break

    if final_pretrain_path is None:
        # Provide comprehensive diagnostic info if file is not found
        contents = {}
        contents['searched_base_paths_considered'] = list(set(possible_paths_to_check)) # All paths we thought of
        contents['searched_base_paths_actually_checked'] = list(checked_paths) # Paths that actually existed and were checked

        # Add actual file system contents for diagnostics for all checked paths
        for p_base_diag in checked_paths:
            if os.path.exists(p_base_diag):
                try:
                    contents[f'contents_of_{p_base_diag}'] = os.listdir(p_base_diag)
                    for entry in os.listdir(p_base_diag):
                        sub_path = os.path.join(p_base_diag, entry)
                        if os.path.isdir(sub_path):
                            try:
                                contents[f'contents_of_subdir_{entry}_in_{p_base_diag}'] = os.listdir(sub_path)
                            except Exception as sub_e:
                                contents[f'contents_of_subdir_{entry}_in_{p_base_diag}_error'] = str(sub_e)
                except Exception as e_dir:
                    contents[f'contents_of_{p_base_diag}_error'] = str(e_dir)
            else:
                contents[f'path_not_exists_{p_base_diag}'] = True # Should not happen for paths in checked_paths

        raise ProjectException(f"Could not locate '{target_file}' in any expected paths. Diagnostic: {json.dumps(contents, indent=2)}")
except Exception as e:
    raise ProjectException("eda failed", e)

In [ ]:
final_pretrain_path

In [ ]:
try:
    eda_pretrain(final_pretrain_path)
except Exception as e:
    raise ProjectException("eda failed", e)

In [ ]:
try:
    eda_qa(qa_dir)
except Exception as e:
    raise ProjectException("eda failed", e)

### Tokenization using GPT-2 Tokenizer

In [ ]:
!pip install transformers -q

In [ ]:
from transformers import GPT2Tokenizer

def load_tokenizer():
    """loads the GPT-2 tokenizer """
    try:
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.model_max_length = 10**9
        #gpt2 tokenizer uses byte-level BPE; no explicit <unk> token by default
        #lets add one for saftey
        tokenizer.add_special_tokens({'unk_token': '<|UNK|>'})
        tokenizer.add_special_tokens({'pad_token': '<|PAD|>'})
        #model embedding matrix will be resized
        print("tokenizer loaded")
        return tokenizer
    except Exception as e:
        raise ProjectException("failed to load tokenizer", e)

#test run
tokenizer = load_tokenizer()

sample_text = (
    "The transformer architecture has revolutionized nlp"
    "It relies on self-attention mechanisms to capture long-range dependencies"
)

encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded)

print("Sample text:", sample_text)
print("\nEncoded token IDs:", encoded)
print("Decoded back:", decoded)
print("Vocabulary size:", tokenizer.vocab_size)
print("Special tokens:", tokenizer.special_tokens_map)
print("EOS token ID:", tokenizer.eos_token_id)

### Pytorch Datasets (pretraining & q&a)

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import GPT2Tokenizer
from typing import List, Dict, Tuple
import json
import os
import math

class PreTrainingDataset(Dataset):
    """
    Generates causal LM sequences from raw text.
    Reads a raw text file (one paragraph per line), tokenizes everything into one long tensor,
    then slices into blocks of `max_seq_len`. Each example is (input_ids, labels) where
    labels == input_ids (shift happens inside the model via causal mask).
    """
    def __init__(self, tokenizer: GPT2Tokenizer, file_path: str, max_seq_len: int = 512):
        self.max_seq_len = max_seq_len
        self.tokenizer = tokenizer
        self.examples = [] #list of tensors (seq_len,)

        #read and tokenize the wholw file
        with open(file_path,"r",encoding="utf-8") as f:
            lines = f.readlines()

        #tokenize each line and flatten into one large list of tokens
        all_tokens = []
        for line in lines:
            line_tokens = tokenizer.encode(line, add_special_tokens=True)
            all_tokens.extend(line_tokens)

        #crete blocks of max_seq_len
        total_tokens = len(all_tokens)
        num_blocks = (total_tokens - 1) // max_seq_len
        for i in range(num_blocks):
            block = all_tokens[i * max_seq_len : (i + 1) * max_seq_len + 1]
            input_ids = torch.tensor(block[:-1], dtype=torch.long)
            labels    = torch.tensor(block[1:],  dtype=torch.long)
            self.examples.append({"input_ids": input_ids, "labels": labels})

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]




class QADataset(Dataset):
    """Fine‑tuning dataset for Q&A. Formats each example as:
    <|question|> question_text <|context|> context_text <|answer|> answer_text <|endoftext|>
    Labels are set to -100 for all prompt tokens, so the loss is computed only on the answer part.
    """
    def __init__(self, json_path:str, tokenizer:GPT2Tokenizer,max_seq_len:int=512) :
        self.tokenizer = tokenizer
        self.examples = []
        self.max_seq_len = max_seq_len
        special_tokens = {
            'question_token': '<|question|>',
            'context_token': '<|context|>',
            'answer_token': '<|answer|>',
            'unk_answer': 'unanswerable'
        }
        tokenizer.add_special_tokens({'additional_special_tokens': list(special_tokens.values())})
        self.question_token_id = tokenizer.convert_tokens_to_ids(special_tokens['question_token'])
        self.context_token_id  = tokenizer.convert_tokens_to_ids(special_tokens['context_token'])
        self.answer_token_id   = tokenizer.convert_tokens_to_ids(special_tokens['answer_token'])
        self.unk_answer_id     = tokenizer.convert_tokens_to_ids(special_tokens['unk_answer'])


        with open(json_path,"r") as f:
            self.qa_data = json.load(f)

        for item in self.qa_data:
            # Safely convert potential float NaNs or None to empty strings
            question = str(item.get("question", "")) if item.get("question") is not None and not (isinstance(item.get("question"), float) and math.isnan(item.get("question"))) else ""
            context = str(item.get("context", "")) if item.get("context") is not None and not (isinstance(item.get("context"), float) and math.isnan(item.get("context"))) else ""
            answer = str(item.get("answer", "")) if item.get("answer") is not None and not (isinstance(item.get("answer"), float) and math.isnan(item.get("answer"))) else ""


            answer_start = item.get("answer_start")
            answer_end = item.get("answer_end")

            prompt = (
                f"{special_tokens['question_token']} {question} "
                f"{special_tokens['context_token']} {context} "
                f"{special_tokens['answer_token']} {answer} "
            )
            full_text = prompt + " " + answer + tokenizer.eos_token

            full_ids = tokenizer.encode(full_text)
            prompt_ids = tokenizer.encode(prompt)
            prompt_len = len(prompt_ids)

            if len(full_ids) > self.max_seq_len + 1:
                full_ids = full_ids[-(self.max_seq_len + 1):]
                try:
                   ans_idx = full_ids.index(self.answer_token_id) + 1
                   prompt_len = ans_idx
                except ValueError:
                   continue

            #shift: input = all tokens except last, labels = all tokens except first
            input_ids = torch.tensor(full_ids[:-1], dtype=torch.long)
            labels = torch.tensor(full_ids[1:], dtype=torch.long)

            #mask akk positions that correspond to prompt tokens in the labels
            # (labels are shifted, so the prompt positions are prompt_len-1 tokens long)
            # The first prompt_len-1 labels should be -100
            mask_len = prompt_len - 1
            if mask_len > 0:
                labels[:mask_len] = -100
            self.examples.append({"input_ids":torch.tensor(full_ids, dtype=torch.long),
                                  "labels":labels.detach().clone() if isinstance(labels, torch.Tensor) else torch.tensor(labels, dtype=torch.long),
                                  "question":question,
                                  "context":context,
                                  "answer_raw":answer})

        print(f"QADataset loaded {len(self.examples)} examples from {json_path}")
    def __getitem__(self, idx):
        return self.examples[idx]

    def __len__(self):
        return len(self.examples)



In [ ]:
# Quick test (ensure path: /content/qa_data.json exists)
qa_dataset = QADataset("/content/qa_data.json", tokenizer, max_seq_len=512)
print("Number of QA examples:", len(qa_dataset))
sample = qa_dataset[0]
print("Sample input_ids (first 30):", sample["input_ids"][:30].tolist())
print("Sample labels (first 30):", sample["labels"][:30].tolist())
print("Decoded input:", tokenizer.decode(sample["input_ids"]))
# Show answer part
answer_mask = sample["labels"] != -100
if answer_mask.any():
    print("Answer text:", tokenizer.decode(sample["labels"][answer_mask]))
else:
    print("No answer tokens found (possibly an unanswerable question).")

In [ ]:
##colate function for dataloader
def collate_fn(batch:List[Dict[str,torch.Tensor]],pad_token_id:int)->Dict[str,torch.Tensor]:
    """
    Collate function to pad sequences to the length of the longest one in the batch.
    """
    input_ids = [item["input_ids"] for item in batch]
    labels = [item["labels"] for item in batch]

    #stack if all lengths identical
    lengths = [t.size(0) for t in input_ids]
    if len(set(lengths)) == 1:
        padded_input_ids = torch.stack(input_ids, dim=0)
        padded_labels = torch.stack(labels, dim=0)
    else:
        #fall back to padding
        padded_input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
        padded_labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)

    #attention mask (1 for real tokens, 0 for padding)
    attention_mask = (padded_input_ids != pad_token_id).long()

    if "question" in batch[0]:
        questions = [item["question"] for item in batch]
        contexts = [item["context"] for item in batch]
        answers = [item["answer_raw"] for item in batch]

        return {
            "input_ids": padded_input_ids,
            "labels": padded_labels,
            "attention_mask": attention_mask,
            "questions": questions,
            "contexts": contexts,
            "answers": answers
        }
    else:
        return {
            "input_ids": padded_input_ids,
            "labels": padded_labels,
            "attention_mask": attention_mask
        }


if tokenizer is not None:
    #locate actual training files in pretrain_dir and qa_dir
    pretrain_train_file = os.path.join(final_pretrain_path, "wiki.train.tokens")
    qa_train_file = os.path.join("/content", "qa_data.json")
    try:
        if os.path.exists(pretrain_train_file) and os.path.exists(qa_train_file):
            pretrain_dataset = PreTrainingDataset(tokenizer, pretrain_train_file, max_seq_len=256)
            qa_dataset = QADataset(qa_train_file, tokenizer, max_seq_len=256)
            print("Number of pretraining examples:", len(pretrain_dataset))
            print("Number of QA examples:", len(qa_dataset))

            #show a sample  from pretraining
            sample_pt = pretrain_dataset[0]
            print("\npretraining sample input_ids (first 30):", sample_pt["input_ids"][:30])
            print("pretraining sample labels (first 30):",sample_pt["labels"][:30])

            #show sample qa
            sample_qa = qa_dataset[0]
            print("\nQA sample input_ids :", sample_qa["input_ids"])
            print("QA sample labels :", sample_qa["labels"])
            print("decoded qa input:", tokenizer.decode(sample_qa["input_ids"]))

            #show where labels are not -100
            answer_mask = sample_qa["labels"] != -100
            print("answer tokens:", tokenizer.decode(sample_qa["labels"][answer_mask]))
        else:
            raise OSError("Could not find files")
    except OSError as e:
        raise ProjectException("Could not find files", e)

In [ ]:
#investigating the collate_fn's job
#any sample that passes through will be padded and the attention mask created


### Dataloaders

In [ ]:
os.cpu_count()

In [ ]:
from torch.utils.data import DataLoader, random_split

BATCH_SIZE_PT = 32
BATCH_SIZE_QA = 32
NUM_WORKERS = os.cpu_count()

from functools import partial

pad_token_id = tokenizer.pad_token_id
collate_fn_pt = partial(collate_fn, pad_token_id=pad_token_id)

#pretrain dataloader
pretrain_file = os.path.join(final_pretrain_path, "wiki.train.tokens")
if not os.path.exists(pretrain_file):
    raise ProjectException(f"Pretraining file not found at {pretrain_file}")

pretrain_dataset = PreTrainingDataset(tokenizer, pretrain_file, max_seq_len=256)

pretrain_dataloader = DataLoader(pretrain_dataset,
                                 batch_size=BATCH_SIZE_PT,
                                 shuffle=True,
                                 num_workers=NUM_WORKERS,
                                 pin_memory=True,
                                 prefetch_factor=4, #prefetch 4 batches per worker
                                 collate_fn=collate_fn_pt)

print(f"Pre-training DataLoader: {len(pretrain_dataloader)} batches "
      f"(batch_size={BATCH_SIZE_PT})")


qa_file = os.path.join("/content/", "qa_data.json")
if not os.path.exists(qa_file):
    raise ProjectException(f"QA file not found at {qa_file}")

qa_dataset = QADataset(qa_file, tokenizer, max_seq_len=128)
qa_dataloader = DataLoader(qa_dataset,
                           batch_size=BATCH_SIZE_QA,
                           shuffle=True,
                           num_workers=NUM_WORKERS,
                           prefetch_factor=6,
                           pin_memory=True,
                           collate_fn=collate_fn_pt)
print(f"QA DataLoader: {len(qa_dataloader)} batches (batch_size={BATCH_SIZE_QA})")

#quick test
pt_batch = next(iter(pretrain_dataloader))
print("\n--- Pre-training batch ---")
print("Pre-training batch keys:", pt_batch.keys())
print("Pre-training batch input_ids shape:", pt_batch["input_ids"].shape)
print("Pre-training batch labels shape:", pt_batch["labels"].shape)
print("attention_mask shape:",pt_batch["attention_mask"].shape)

#view the content of the batch sample
#original input
print("Sample from the batch:", pt_batch["input_ids"][0])
#attention mask
print("Attention mask:", pt_batch["attention_mask"][0])
#decoded input
decoded_pt_input = tokenizer.decode(pt_batch["input_ids"][0])
print("\nDecoded pre-training input (first example):")
print(decoded_pt_input)

qa_batch = next(iter(qa_dataloader))
print("\n--- QA batch ---")
print("QA batch keys:", qa_batch.keys())
print("QA batch input_ids shape:", qa_batch["input_ids"].shape)
print("QA batch labels shape:", qa_batch["labels"].shape)
print("attention_mask shape:",qa_batch["attention_mask"].shape)

# Show a decoded QA example (first in batch)
decoded_qa_input = tokenizer.decode(qa_batch["input_ids"][0])
print("\nDecoded QA input (first example):")
print(decoded_qa_input)

# Verify that labels are -100 for prompt and non‑-100 for answer
first_labels = qa_batch["labels"][0]
answer_mask = first_labels != -100
print("\nNumber of answer tokens:", answer_mask.sum().item())
print("Answer text:", tokenizer.decode(first_labels[answer_mask]))

### MultiHead Self Attention(Causal)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
    """
    Args:
        query:  (batch, num_heads, seq_len, d_k)
        key:    (batch, num_heads, seq_len, d_k)
        value:  (batch, num_heads, seq_len, d_v)
        mask:   (batch, 1, seq_len, seq_len) or broadcastable; True means "ignore"
        dropout: nn.Dropout or None
    Returns:
        output: (batch, num_heads, seq_len, d_v)
        attn_weights: (batch, num_heads, seq_len, seq_len)
    """
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2,-1)) / math.sqrt(d_k) #matrix multiplication of the key and query gives the scores

    if mask is not None:
        #mask filled with True -> set to -inf
        scores = scores.masked_fill(mask, float("-inf"))

    attn_weights = F.softmax(scores,dim=-1)
    if dropout is not None:
        attn_weights = dropout(attn_weights)

    output = torch.matmul(attn_weights,  value)
    return output, attn_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1): #d_model=dimension of embedding model
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads #if it is not integer divisble the assertion will raise

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
    def forward(self,x, causal_mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            causal_mask: (batch, 1, seq_len, seq_len) or broadcastable; True means "ignore"
        Returns:
            output: (batch, seq_len, d_model)
            attn_weights: (batch, num_heads, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.size()

        #linear  projections and reshape for multi-head
        Q = self.W_q(x).view(batch_size,seq_len,self.num_heads, self.d_k).transpose(1,2) #(B,num_heads, T, d_k)
        K = self.W_k(x).view(batch_size,seq_len,self.num_heads, self.d_k).transpose(1,2)
        V = self.W_v(x).view(batch_size,seq_len,self.num_heads, self.d_k).transpose(1,2)

        #scaled dot product attention
        attn_output, attn_weights = scaled_dot_product_attention(Q,K,V, mask=causal_mask, dropout=self.dropout)

        #concatenate heads
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)

        #linear projection
        output = self.W_o(attn_output)
        return output, attn_weights


scaled_dot_product_attention: Computes
softmax
(
Q
K
T
/
d
k
)
V
softmax(QK
T
 /
d
k
​

​
 )V. If a mask is provided (with True meaning “ignore”), it sets those positions to
−
∞
−∞ before softmax. Returns both the output and the attention weights (useful for visualisation).

MultiHeadAttention: Splits the input into num_heads projections, applies the attention function in parallel, then concatenates the heads and passes through a final linear layer (W_o).

Causal masking will be created later in the decoder block (a lower‑triangular matrix of False values). For now, the module accepts an optional causal_mask argument.

### PositionWise Feedforward network

In [ ]:
class PositionWiseFeedForward(nn.Module):
    """
    Two-layer fully connected network with GELU activation,
    applied to each position independently.
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            (batch, seq_len, d_model)
            """
        output = self.fc1(x)
        output = self.gelu(output)
        output = self.dropout(output)
        output = self.fc2(output)
        return output

### Layer Normalization

In [ ]:
class LayerNormalization(nn.Module):
    """
    Custom Layer Normalization (no external implementation).
    Normalises the last dimension of the input.
    """
    def __init__(self, d_model, eps=1e-12):
        super(LayerNormalization, self).__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        #x: (batch, seq_len, d_model)
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True,unbiased=False)
        output = (x - mean) / (std + self.eps)
        output = self.gamma * output + self.beta
        return output

### Transformer Decoder Block

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """
    A single decoder layer with:
      - Pre‑layer norm (GPT‑style)
      - Masked multi‑head self‑attention
      - Position‑wise feed‑forward
      - Residual connections with dropout
    """
    def __init__(self,d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerDecoderBlock, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNormalization(d_model)
        self.norm2 = LayerNormalization(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, causal_mask):
        """
        x: (batch, seq_len, d_model)
        causal_mask: (batch, 1, seq_len, seq_len) or broadcastable
        Returns:
            output: (batch, seq_len, d_model)
            attn_weights: (batch, num_heads, seq_len, seq_len)   (for visualisation)
        """
        #self-attention with residual
        attn_out, attn_weights = self.self_attn(self.norm1(x), causal_mask)
        x = x + self.dropout(attn_out)

        #feedfprward with reidual
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout(ff_out)

        return x, attn_weights

Pre‑norm architecture: LayerNorm is applied before each sub‑layer (attention and feed‑forward), which is standard in GPT‑2 and later models. This stabilises training and allows deeper networks.

The attention sub‑layer receives the causal mask (created outside) and returns both the transformed sequence and the attention weights (for potential analysis).

Dropout is applied to the output of each sub‑layer before the residual addition.

The block is self‑contained; stacking multiple blocks will form the full decoder.

### Full Model

In [ ]:
class GPT(nn.Module):
    """
    Decoder‑only Transformer (GPT‑style) built from scratch.
    """
    def __init__(self,vocab_size,d_model,num_heads,max_seq_len,d_ff,num_layers,dropout=0.1,pad_token_id=None):
        super(GPT, self).__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len,d_model)
        self.blocks = nn.ModuleList([TransformerDecoderBlock(d_model,
                                                             num_heads,
                                                             d_ff,
                                                             dropout)
        for _ in range(num_layers)])
        self.norm = LayerNormalization(d_model)
        self.dropout = nn.Dropout(dropout)
        self.lm_head = nn.Linear(d_model, vocab_size,bias=False)
        self.max_seq_len = max_seq_len
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id

        #weight tying
        self.token_embedding.weight = self.lm_head.weight

        #initialize parameters
        self._init_weights()
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self,input_ids, attention_mask=None):
        """
        Args:
            input_ids: (batch, seq_len)
            attention_mask: (batch, seq_len) with 1 for real tokens, 0 for pad
        Returns:
            logits: (batch, seq_len, vocab_size)
            attn_weights_list: list of attention weight tensors from each block
        """
        B,T = input_ids.shape
        assert T <= self.max_seq_len, f"seq len {T} > max_seq_len {self.max_seq_len}"

        #token + position embeddings
        tok_emb = self.token_embedding(input_ids) #(B,T,d_model)
        positions = torch.arange(0,T, device=input_ids.device).unsqueeze(0).expand(B,T)
        pos_emb = self.position_embedding(positions)

        x = self.dropout(tok_emb + pos_emb)

        #causal mask: lower triangualr = Fasle (allow), upper=True (mask)
        causal_mask = torch.triu(
            torch.ones(T,T, device=input_ids.device, dtype=torch.bool),
            diagonal=1
        )
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        #combine with padding mask if needed
        if attention_mask is not None and self.pad_token_id is not None:
            #padding positions are True (masked)
            pad_mask = (attention_mask == 0).unsqueeze(1).unsqueeze(2) #(B,1,1,T)
            causal_mask = causal_mask | pad_mask

        #transformer blocks
        attn_weights_list = []
        for block in self.blocks:
            x, attn_weights = block(x, causal_mask)
            attn_weights_list.append(attn_weights)
        x = self.norm(x)
        logits = self.lm_head(x) #(B,T, vocab_size)
        return logits, attn_weights_list
    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters())

#instantiate a small model for testing
MODEL_CONFIG = {
    "vocab_size": len(tokenizer),          # len(tokenizer) accounts for added special tokens
    "d_model": 768, #embedding dimensions used
    "num_heads": 12,
    "num_layers": 12,
    "d_ff": 3072,                                # 4 × d_model
    "max_seq_len": 512,
    "dropout": 0.1,
    "pad_token_id": tokenizer.pad_token_id
}

model = GPT(**MODEL_CONFIG)
print(f"Number of parameters: {model.get_num_parameters():_} MB")

#test forward pass with a dummy batch from pretrain_dataloader
dummy_batch = next(iter(pretrain_dataloader))
with torch.no_grad():
    logits, attn_weights_list = model(dummy_batch["input_ids"], dummy_batch["attention_mask"])
print("\n--- Forward pass ---")
print("logits shape:", logits.shape)
print("attn_weights_list length:", len(attn_weights_list))

The logits shape torch.Size([8, 128, 50263]) represents the output of the GPT model's final linear layer (self.lm_head). Let's break down each dimension:

8: This is the batch size. It means that 8 sequences were processed simultaneously in this particular forward pass.

128: This is the sequence length (T), or max_seq_len for this batch. It indicates that each sequence in the batch had 128 tokens.

50263: This is the vocabulary size. It represents the total number of unique tokens (words, subwords, special tokens) that your tokenizer and model are trained to recognize and generate. For each position in each sequence, the model outputs a probability distribution over all possible tokens in the vocabulary.

So, logits[b, t, v] gives you the unnormalized log-probabilities (scores) for each vocabulary token v at position t in batch b. These logits are then typically passed through a softmax function to get actual probabilities, and then used with CrossEntropyLoss for training.

### Losses or Compute Loss Function

In [ ]:
def compute_loss(logits, labels,  ignore_index=-100):
    """
    Standard causal LM loss: cross‑entropy over vocabulary.
    The model outputs logits for all positions; labels are the target token IDs.
    Positions with label == ignore_index are excluded from the loss.
    """
    #flatten to (B*T, vocab_size) and (B*T)
    loss_fn = nn.CrossEntropyLoss(ignore_index=ignore_index)
    loss = loss_fn(logits.view(-1,logits.size(-1)), labels.view(-1))
    return loss

#lets test with the dummy
with torch.no_grad():
    loss = compute_loss(logits, dummy_batch["labels"])
print("\n--- Loss ---")
print("loss:", loss)

### Visualize the attention and masking

In [ ]:
import matplotlib.pyplot as plt
import torch

def create_combined_mask(seq_len, pad_mask=None):
    """
    Returns a boolean mask of shape (seq_len, seq_len) where True means "ignore".
    If pad_mask is given (1=real, 0=pad), positions corresponding to pad tokens are masked.
    """
    causal = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)

    if pad_mask is not None:
        #pad_mask shape: (seq_len,),0 for padding
        pad = (pad_mask == 0).unsqueeze(0)
        causal = causal | pad
    return causal

def plot_mask(mask, annotate_values=True,title="Attention MAsk"):
    plt.figure(figsize=(6,5))
    plt.imshow(mask, cmap='Blues_r', interpolation='nearest')
    plt.title(title)
    plt.xlabel("Key position")
    plt.ylabel("Query position")
    if annotate_values:
        for i in range(seq_len):
            for j in range(seq_len):
                val = "T" if mask[i,j] else "F"
                color = "white" if mask[i,j] else "black"
                plt.text(j, i, val, ha="center", va="center", color="black")
    plt.colorbar(label="T = masked")
    plt.show()

def plot_attention_scores_after_masking(seq_len, pad_mask=None):
    """
    Generate random attention scores (like QK^T) and show the values
    after applying the mask: unmasked = original value, masked = -inf.
    """
    scores = torch.randn(seq_len, seq_len) * 2   # random scores
    mask = create_combined_mask(seq_len, pad_mask)
    scores_masked = scores.masked_fill(mask, float('-inf'))

    # Convert to numpy for plotting
    arr = scores_masked.numpy()
    arr_vis = np.where(np.isneginf(arr), -1e9, arr)  # for visualisation color scale

    plt.figure(figsize=(max(6, seq_len*0.6), max(5, seq_len*0.5)))
    plt.imshow(arr_vis, cmap='viridis', interpolation='nearest')
    plt.title("Attention scores after masking (-inf shown)")
    plt.xlabel("Key position")
    plt.ylabel("Query position")
    # Annotate with actual values (show "-inf" for masked)
    for i in range(seq_len):
        for j in range(seq_len):
            if np.isneginf(arr[i, j]):
                text = "-inf"
                color = "white"
            else:
                text = f"{arr[i, j]:.1f}"
                color = "black"
            plt.text(j, i, text, ha="center", va="center", color=color, fontsize=7)
    plt.colorbar()
    plt.show()
#test
# pure causal mask (no padding)
seq_len = 10
mask_causal = create_combined_mask(seq_len)
plot_mask(mask_causal, title="Pure Causal Mask")

#causal + padding_mask (dimulate a sequence with last 3 tokens as padding)
pad_mask_example = torch.ones(seq_len)
pad_mask_example[-3:] = 0 #last 3 are padding
mask_causal_pad = create_combined_mask(seq_len, pad_mask=pad_mask_example)
plot_mask(mask_causal_pad, title="Causal + Padding Mask")

print("\nActual attention scores after masking (no padding)")
plot_attention_scores_after_masking(seq_len)

print("\nActual attention scores after masking (with padding at end)")
plot_attention_scores_after_masking(seq_len, pad_mask_example)


### Training Loop

In [ ]:
!pip install mlflow -q

In [ ]:
##check cuda
!nvidia-smi

In [ ]:
##check what version of pytorch iam using and if it pytorch cuda
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
import torch

# 1. Current memory
print("GPU memory allocated:", torch.cuda.memory_allocated() / 1e9, "GB")
print("GPU memory cached:", torch.cuda.memory_reserved() / 1e9, "GB")

# 2. Check utilisation (run nvidia-smi from shell)
!nvidia-smi --query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total --format=csv


In [ ]:
#get training resource info
import torch
def get_resource_name():
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_count = torch.cuda.device_count()
        return f"GPU: {gpu_name} ({gpu_count} devices)"
    else:
        return "CPU"
resource_name = get_resource_name()
print(f"Resource: {resource_name}")

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8000)

In [ ]:
import torch

print("Before clearing cache:")
print("GPU memory allocated:", torch.cuda.memory_allocated() / 1e9, "GB")
print("GPU memory cached:", torch.cuda.memory_reserved() / 1e9, "GB")

torch.cuda.empty_cache()

print("\nAfter clearing cache:")
print("GPU memory allocated:", torch.cuda.memory_allocated() / 1e9, "GB")
print("GPU memory cached:", torch.cuda.memory_reserved() / 1e9, "GB")

# You can also run nvidia-smi again to verify
print("\nRunning nvidia-smi again:")
!nvidia-smi --query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total --format=csv

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from warnings import WarningMessage
import time
import math
import mlflow
import torch
from torch.cuda.amp import GradScaler, autocast
from tqdm.auto import tqdm
import os

#configs to monitor gpu during training
TARGET_EFFECTIVE_BATCH_SIZE = 64
#auto compute graient accumulation to reach target
MICRO_BATCH_SIZE = BATCH_SIZE_PT
assert TARGET_EFFECTIVE_BATCH_SIZE % MICRO_BATCH_SIZE == 0, "effective batch size should be a multiple of micro batch size"
GRAD_ACCUM_STEPS = TARGET_EFFECTIVE_BATCH_SIZE // MICRO_BATCH_SIZE

print(f"Effective batch size: {TARGET_EFFECTIVE_BATCH_SIZE}")
print(f"micro batch size: {MICRO_BATCH_SIZE} -> accmulate {GRAD_ACCUM_STEPS} steps")

#mixed precision
SCALER = torch.amp.GradScaler('cuda') # Updated GradScaler call

#mlflow control
USE_MLFLOW = True

#download best pth
DOWNLOAD_BEST_MODEL = True

#device

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

#model to device
model.to(DEVICE)

#validation dataset and loader (using wiki.test.tokens)
pretrain_test_file = os.path.join(final_pretrain_path, "wiki.test.tokens")
if not os.path.exists(pretrain_test_file):
    raise ProjectException(f"Pretrain test file not found: {pretrain_test_file}")

valid_dataset_file = PreTrainingDataset(file_path=pretrain_test_file,
                                        tokenizer=tokenizer,
                                        max_seq_len=MODEL_CONFIG["max_seq_len"])

valid_dataloader = DataLoader(valid_dataset_file,
                              batch_size=BATCH_SIZE_PT,
                              shuffle=False,
                              num_workers=NUM_WORKERS,
                              collate_fn=collate_fn_pt,
                              pin_memory=True,
                              prefetch_factor=2)

#mlflow setup
MLFLOW_TRACKING_URI = "http://0.0.0.0:8000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("decoder_only_transformer_pretrain")

#hyperparameters (use MODEL_CONFIG where possible)
params = {
    "batch_size_train": BATCH_SIZE_PT ,
    "learning_rate": 1e-2,
    "num_epochs": 25,
    "weight_decay":0.001,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "resource": resource_name
}

#setup optimizer
optimizer = torch.optim.AdamW(model.parameters(),
                              lr=params["learning_rate"],
                              weight_decay=params["weight_decay"])

#sceduler-cosine annealing with warmup
total_steps = len(pretrain_dataloader) * params["num_epochs"] // params["grad_accum_steps"]
warmup_steps = int(0.1 * total_steps)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=params["learning_rate"],
    total_steps=total_steps,
    pct_start=warmup_steps / total_steps,
    div_factor=100,
    final_div_factor=100,
    anneal_strategy="cos",
)

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pth")

#helper function to compute perplexity
def compute_perplexity(loss):
    try:
        return math.exp(loss)
    except OverflowError:
        return float("inf")

def train_epoch_amp(model, dataloader, optimizer, scaler, device, scheduler, grad_accum_steps=1):
    model.train()
    total_loss_sum = 0.0          # sum of loss * number of target tokens
    total_tokens = 0
    pbar = tqdm(dataloader, desc="Training", leave=True)

    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.amp.autocast('cuda'):
            logits, _ = model(input_ids, attention_mask)
            loss = compute_loss(logits, labels)        # already mean over non‑ignored tokens
            loss = loss / grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        # Accumulate loss weighted by the number of non‑ignored tokens in this batch
        batch_loss_value = loss.item() * grad_accum_steps
        non_ignored = (labels != -100).sum().item()   # for pre‑training this equals all tokens
        total_loss_sum += batch_loss_value * non_ignored
        total_tokens += non_ignored

        # Live perplexity from the running average loss per token
        avg_loss = total_loss_sum / total_tokens if total_tokens > 0 else 0.0
        current_ppl = math.exp(avg_loss) if avg_loss < 100 else float('inf')

        if step % 100 == 0:
            mem_used = torch.cuda.memory_allocated() / 1e9
            pbar.set_postfix({
                "loss": batch_loss_value,
                "avg_loss": avg_loss,
                "ppl": current_ppl,
                "lr": scheduler.get_last_lr()[0],
                "GPU": f"{mem_used:.1f}GB"
            })
        else:
            pbar.set_postfix({
                "loss": batch_loss_value,
                "ppl": current_ppl,
                "lr": scheduler.get_last_lr()[0]
            })

    # Epoch‑level metrics: average loss over all tokens seen
    epoch_avg_loss = total_loss_sum / total_tokens
    epoch_ppl = math.exp(epoch_avg_loss) if epoch_avg_loss < 100 else float('inf')
    return epoch_avg_loss, epoch_ppl


@torch.no_grad()
def validate(model, dataloader, device):
    model.eval()
    total_loss_sum = 0.0
    total_tokens = 0
    pbar = tqdm(dataloader, desc="Validation", leave=True)

    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.amp.autocast('cuda'):
            logits, _ = model(input_ids, attention_mask)
            loss = compute_loss(logits, labels)

        batch_loss = loss.item()
        non_ignored = (labels != -100).sum().item()
        total_loss_sum += batch_loss * non_ignored
        total_tokens += non_ignored

        avg_loss = total_loss_sum / total_tokens if total_tokens > 0 else 0.0
        ppl = math.exp(avg_loss) if avg_loss < 100 else float('inf')
        pbar.set_postfix({"loss": batch_loss, "avg_loss": avg_loss, "ppl": ppl})

    epoch_avg_loss = total_loss_sum / total_tokens
    epoch_ppl = math.exp(epoch_avg_loss) if epoch_avg_loss < 100 else float('inf')
    return epoch_avg_loss, epoch_ppl

def pretrain():
    best_val_ppl = float("inf")
    early_stopping_patience = 2
    patience_counter = 0

    run = None
    if USE_MLFLOW:
        try:
            run = mlflow.start_run()
            #log hyperparameters with the model configs too
            mlflow.log_params(
            {
            "d_model": MODEL_CONFIG["d_model"],
                "num_heads": MODEL_CONFIG["num_heads"],
                "num_layers": MODEL_CONFIG["num_layers"],
                "d_ff": MODEL_CONFIG["d_ff"],
                "max_seq_len": MODEL_CONFIG["max_seq_len"],
                "dropout": MODEL_CONFIG["dropout"],
                "micro_batch_size": MICRO_BATCH_SIZE,
                "effective_batch_size": TARGET_EFFECTIVE_BATCH_SIZE,
                "grad_accum_steps": GRAD_ACCUM_STEPS,
                "epochs": params['num_epochs'],
                "lr": params["learning_rate"],
                "vocab_size": MODEL_CONFIG["vocab_size"],
                "mixed_precision": True
            })

            for epoch in range(1, params.get("num_epochs",5)+1):

                print(f"\n==Epoch {epoch}/{params.get("num_epochs",5)}==")

                train_loss, train_ppl = train_epoch_amp(model, pretrain_dataloader, optimizer, SCALER, DEVICE, scheduler, params.get("grad_accum_steps",1))
                print(f"Train loss: {train_loss:.4f}, Train PPL: {train_ppl:.2f}")

                val_loss, val_ppl = validate(model, valid_dataloader,DEVICE)
                print(f"Val loss: {val_loss:.4f}, Val PPL: {val_ppl:.2f}")

                #log metrics with mlflow
                mlflow.log_metrics({"train_loss": train_loss,
                                "train_ppl": train_ppl,
                                "val_loss": val_loss,
                                "val_ppl": val_ppl},
                               step=epoch)
                #overfitting check: early stopping if validation ppl increases
                if val_ppl < best_val_ppl:
                    best_val_ppl = val_ppl
                    patience_counter = 0
                    torch.save(model.state_dict(), BEST_MODEL_PATH)
                    #download the best pth
                    #from google.colab import files
                    #if DOWNLOAD_BEST_MODEL:
                        #files.download(BEST_MODEL_PATH)
                    print(f"saved best model (ppl={best_val_ppl:.2f})")
                else:
                    patience_counter += 1
                    print(f"validation ppl didnt improve. patience: {patience_counter}/{early_stopping_patience}")
                    if patience_counter >= early_stopping_patience:
                        print(f"Early stopping triggered. Best val ppl: {best_val_ppl:.2f}")
                        break
            #final checkpoint (last epoch)
            final_path = os.path.join(CHECKPOINT_DIR, "final_model.pth")
            torch.save(model.state_dict(), final_path)
            print(f"saved final model with (ppl={best_val_ppl:.2f}) to {final_path}")

            #log model artefact
            mlflow.log_artefact(BEST_MODEL_PATH)
        except Exception as e:
            raise ProjectException(f"Error during pretraining: {e}")
        finally:
            if run is not None:
                mlflow.end_run()


#setup mlflow server
#!mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns --host 0.0.0.0 --port 8000 --allowed-hosts "*" --cors-allowed-origins "*" &

#call pretrain
pretrain()

### Evaluation Phase

In [ ]:
import math
import torch
from torch.cuda.amp import GradScaler, autocast
from tqdm.auto import tqdm
import os

#load the best pretrained model
print("Loading best pre‑trained checkpoint...")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE)
print("checkpoint loaded..")

#qa dataset and aplit
qa_json_path = "/content/qa_data.json"
full_qs_dataset = QADataset(json_path=qa_json_path, tokenizer=tokenizer,max_seq_len=MODEL_CONFIG["max_seq_len"])

#90/10 train-val split
train_size = int(0.9 * len(full_qs_dataset))
val_size = len(full_qs_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_qs_dataset, [train_size, val_size])
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

#dataloaders
QA_MICRO_BATCH = 16
QA_TARGET_EFFECTIVE = 32
QA_GRAD_ACCUM_STEPS = QA_TARGET_EFFECTIVE // QA_MICRO_BATCH

qa_train_loader = DataLoader(train_dataset,
                              batch_size=QA_MICRO_BATCH,
                              shuffle=True,
                              num_workers=NUM_WORKERS,
                              collate_fn=collate_fn_pt,
                             pin_memory = True,
                             prefetch_factor=2)

qa_val_loader = DataLoader(val_dataset,
                            batch_size=QA_MICRO_BATCH,
                            shuffle=False,
                            num_workers=NUM_WORKERS,
                            collate_fn=collate_fn_pt,
                            pin_memory = True,
                            prefetch_factor=2)

#optimizer & scheduler (lower LR for finetuning)
FT_LR = 1e-4
FT_EPOCHS = 5
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR)
total_steps = len(qa_train_loader) * FT_EPOCHS // QA_GRAD_ACCUM_STEPS
warmup_steps = int(0.1 * total_steps)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=FT_LR,
    total_steps=total_steps,
    pct_start=warmup_steps / total_steps,
    div_factor=100,
    final_div_factor=100,
    anneal_strategy="cos",
)
SCALER = torch.amp.GradScaler('cuda')

BEST_QA_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_finetuned_qa_model.pth")

#training functions (identical structure, adapted for QA)
def compute_perplexity(loss):
    try:
        return math.exp(loss)
    except OverflowError:
        return float("inf")

def train_qa_epoch(model, dataloader, optimizer, scaler, device, scheduler, grad_accum_steps=1):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    pbar = tqdm(dataloader, desc="Training",leave=True)
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.amp.autocast('cuda'):
            logits, _ = model(input_ids, attention_mask)
            loss = compute_loss(logits, labels) / grad_accum_steps
        scaler.scale(loss).backward()

        if (step + 1) % grad_accum_steps == 0:
            scaler.unscale(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step()
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        batch_loss = loss.item() * grad_accum_steps
        #count answer tokens only (where labels != -100)
        answer_tokens = (labels != -100).sum().item()
        total_loss += batch_loss * input_ids.size(0)
        total_tokens += answer_tokens

        current_perplexity = compute_perplexity(total_loss / len(dataloader.dataset)) if total_tokens > 0  else 0.0

        pbar.set_postfix(
            {
                "train_loss": batch_loss,
                "train_ppl": current_perplexity,
                "train_lr": scheduler.get_last_lr()[0],
            }
        )
    avg_loss = total_loss / len(dataloader.dataset)
    ppl = compute_perplexity(avg_loss)
    return avg_loss, ppl

@torch.no_grad()
def validate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    for batch in tqdm(dataloader, desc="Validation",leave=True):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast():
            logits,_ = model(input_ids, attention_mask)
            loss = compute_loss(logits, labels)

        total_loss += loss.item() * input_ids.size(0)
    avg_loss = total_loss / len(dataloader.dataset)
    ppl = compute_perplexity(avg_loss)
    return avg_loss, ppl

#main fine-tuning loop
def finetune_qa():
    best_val_ppl = float("inf")
    early_stop_patience = 2
    patience_counter = 0

    for epoch in range(1,FT_EPOCHS+1):
        print(f"\n==Epoch {epoch}/{FT_EPOCHS}==")
        train_loss, train_ppl = train_qa_epoch(model, qa_train_loader, optimizer, SCALER, DEVICE, scheduler, QA_GRAD_ACCUM_STEPS)
        print(f"Train loss: {train_loss:.4f}, Train PPL: {train_ppl:.2f}")

        val_loss, val_ppl = validate(model, qa_val_loader, DEVICE)
        print(f"Val loss: {val_loss:.4f}, Val PPL: {val_ppl:.2f}")

        if val_ppl < best_val_ppl:
            best_val_ppl = val_ppl
            patience_counter = 0
            torch.save(model.state_dict(), BEST_QA_MODEL_PATH)
            print(f"saved best model (ppl={best_val_ppl:.2f})")
        else:
            patience_counter += 1
            print(f"validation ppl didnt improve. patience: {patience_counter}/{early_stop_patience}")
            if patience_counter >= early_stop_patience:
                print(f"Early stopping triggered. Best val ppl: {best_val_ppl:.2f}")
                break
    #final save
    final_qa_path = os.path.join(CHECKPOINT_DIR, "final_finetuned_qa_model.pth")
    torch.save(model.state_dict(), final_qa_path)
    print(f"saved final model with (ppl={best_val_ppl:.2f}) to {final_qa_path}")
    print("finetuning complete")


finetune_qa()

### Inference Phase

In [ ]:
import torch
import torch.nn.functional as F


#loading finetuned model
model.load_state_dict(torch.load(BEST_QA_MODEL_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE)
model.eval()


def top_k_to_p_filter(logits, top_k=0, top_p=0.0, filter_value=-float("Inf")):
    """
    Filter logits using top‑k and/or nucleus (top‑p) filtering.
    logits: (batch_size, vocab_size)
    """
    if top_k > 0:
        top_k = min(top_k, logits.size(-1))
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = filter_value

    if top_p > 0.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        #remove tokens with cumulative prob above the threshold
        sorted_indices_to_remove = cumulative_probs > top_p
        #shift the indices to the right to keep also the first token above the threshold
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices_to_remove.scatter(dim=1, index=sorted_indices, src=sorted_indices_to_remove)
        logits[indices_to_remove] = filter_value
    return logits

@torch.no_grad()
def generate_text(
        model, tokenizer, prompt, max_new_tokens=50, temperature=1.0,
        top_k=0, top_p=0.0, device="cuda"):
    """
    Generate text from a prompt.
    Supports greedy (temperature=0), sampling with temperature, top‑k, top‑p.
    """
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids
    for _ in range(max_new_tokens):
        #take only the last "max_seq_len" tokens to avoif OOM
        if generated.size(0) > MODEL_CONFIG["max_seq_len"]:
            input_ids = generated[:, -MODEL_CONFIG["max_seq_len"]:]
        #forward pass (no attention mask needed for single sequence if no padding)
        logits, _ = model(input_ids, attention_mask=None)
        next_token_logits = logits[:,-1,:] / max(temperature, 1e-7)

        #apply top_k / top_p filter
        next_token_logits = top_k_to_p_filter(next_token_logits, top_k=top_k, top_p=top_p)

        #sample or greedy
        if temperature == 0.0:
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.argmax(probs, dim=-1, keepdim=True)
        else:
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        generated = torch.cat([generated, next_token], dim=1)

        #stop if EOS token generated
        if next_token.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(generated[0], skip_special_tokens=False) #keep at false foe better decoding


#qa specific generation
def answer_question(model, tokenizer, question, context, max_new_tokens=30, temperature=0.0, device="cuda"):
    """
    Format the prompt with special tokens and generate an answer.
    """
    prompt = f"<|question|>{question}<|context|>{context}<|answer|>"
    answer = generate_text(model, tokenizer, prompt, max_new_tokens=max_new_tokens, temperature=temperature, device=device)


#demo
print("="*50)

print("open ended generation (pre-trained) - greedy:")

prompt_pt = "once upon a time, in a land far away,"

#load the pretrained model to aboid mixing tasks
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE)
print(generate_text(model, tokenizer, prompt_pt, max_new_tokens=50, temperature=0.0, device=DEVICE))

print("\nOpen_ended generation - sampling with top-k=50, temperature=0,8:")
print(generate_text(model,tokenizer, prompt_pt, max_new_tokens=50,temperature=0.8,top_k=50, device=DEVICE))

#qa generation (load fine-tuned model)
model.load_state_dict(torch.load(BEST_QA_MODEL_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE)

#give the question
question = "To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?"

#context
context = ("Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. "
           "Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend 'Venite Ad Me Omnes'. "
           "Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. "
           "It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858."
           )

answer = answer_question(model, tokenizer, question, context, max_new_tokens=50, temperature=0.0, device=DEVICE)

print("Q:", question)
print("A:", answer)

#unanswerable question
q2 = "what is the capital of Mars?"
ctx2 = "mars is a planet in our solar system. It has no known cities"
print("\nQ:",q2)
print("A:",answer_question(model, tokenizer, q2, ctx2, max_new_tokens=50, temperature=0.0, device=DEVICE))

##Evaluation (Perplexity + Q&A Metrics)

In [ ]:
##for this i will use the bleu_score for QA performance
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
import re

#perplexity evaluation
@torch.no_grad()
def evaluate_perplexity(model, dataloader, device,desc="Eval"):
    model.eval()
    total_loss = 0.0
    for batch in tqdm(dataloader, desc="Evaluation",leave=True):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits, _ = model(input_ids, attention_mask)
        loss = compute_loss(logits, labels)
        total_loss += loss.item() * input_ids.size(0)
    avg_loss = total_loss / len(dataloader.dataset)
    ppl = math.exp(avg_loss) if avg_loss < 100 else float("inf")
    return avg_loss,ppl

#pretraining test set
print("==pre-training test perplexity==")
pt_test_file = os.path.join(final_pretrain_path, "wiki.test.tokens")
if not os.path.exists(pt_test_file):
    raise ProjectException("test file not found")

pt_test_dataset = PreTrainingDataset(pt_test_file, tokenizer, max_seq_len=MODEL_CONFIG["max_seq_len"])
pt_test_loader = DataLoader(pt_test_dataset,
                            batch_size=MICRO_BATCH_SIZE,
                            shuffle=False,
                            num_workers=NUM_WORKERS,
                            collate_fn=collate_fn_pt,
                            pin_memory = True,
                            prefetch_factor=2)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))
model.to(DEVICE)
pt_test_loss, pt_test_ppl = evaluate_perplexity(model, pt_test_loader, DEVICE,desc="pre-train test")
print(f"Test loss: {pt_test_loss:.4f}, Test PPL: {pt_test_ppl:.2f}")


# QA validation perplexity (using the fine‑tuned model)
print("\n=== QA Fine‑tuned Validation Perplexity ===")
model.load_state_dict(torch.load(BEST_QA_MODEL_PATH, map_location=DEVICE,weights_only=True))
model.to(DEVICE)
qa_val_loss, qa_val_ppl = evaluate_perplexity(model, qa_val_loader, DEVICE, desc="QA val")
print(f"QA Val loss: {qa_val_loss:.4f}, Perplexity (answer only): {qa_val_ppl:.2f}")

In [ ]:
#qa text generation
def normalize_text(s):
    """Lowercase, strip, remove extra whitespace."""
    s = str(s).lower().strip()
    s = re.sub(r"\s+", " ",s)
    return s

def compute_qa_metrics(model,tokenizer, dataset, device, max_samples=200, max_new_tokens=50):
    """
    Generate answers for a subset of the dataset and compute BLEU, ROUGE, EM.
    dataset should be a QADataset (or its subset) returning raw texts.
    """
    model.eval()
    model.to(device)
    references = []
    predictions = []
    em_count = 0
    n = min(max_samples, len(dataset))

    for idx in tqdm(range(n),desc="generating answers"):
        sample = dataset[idx]
        question = sample["question"]
        context = sample["context"]
        answer_ref = sample["answer_raw"] #orifinal answer string

        prompt = f"<|question|>{question}<|context|>{context}<|answer|>"
        answer_gen = generate_text(model,
                                   tokenizer,
                                   prompt,
                                   max_new_tokens=max_new_tokens,
                                   temperature=0.0,
                                   device=device)

        #the generated answer is the part after the prompt
        #we already have the whole generated text; extract just the answer
        answer_start = answer_gen.find("<|answer|>") + len("<|answer|>")
        pred_ans = answer_gen[answer_start:].strip() #picks from where the answer start token is bcos its followed by the answer
        if answer_gen == -1:
            raise ProjectException(f"answer not found in generated text: {answer_gen}")
        #normalize
        norm_pred = normalize_text(pred_ans)
        norm_ref = normalize_text(answer_ref)
        references.append([norm_ref.split()]) # for BLEU: list of tokenized references
        predictions.append([norm_pred.split()])
        if norm_pred == norm_ref:
            em_count += 1
    #BLEU
    smoothie = SmoothingFunction().method1
    bleu_score = []
    for ref, pred in zip(references, predictions):
        bleu_score.append(sentence_bleu(ref, pred, smoothing_function=smoothie))
    avg_bleu = sum(bleu_score) / len(bleu_score)
    #ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge1_f1, rouge2_f1, rougeL_f1 = 0.0,0.0,0.0
    for ref_tokens, pred_tokens in zip(references, predictions):
        ref_str = " ".join(ref_tokens[0]) if ref_tokens else ""
        pred_str = " ".join(pred_tokens[0])
        scores = scorer.score(ref_str, pred_str)
        rouge1_f1 += scores['rouge1'].fmeasure
        rouge2_f1 += scores['rouge2'].fmeasure
        rougeL_f1 += scores['rougeL'].fmeasure
    avg_rouge1 = rouge1_f1 / len(references)
    avg_rouge2 = rouge2_f1 / len(references)
    avg_rougeL = rougeL_f1 / len(references)

    #EM
    em_score = em_count / n
    return{
        "bleu": avg_bleu,
        "rouge1": avg_rouge1,
        "rouge2": avg_rouge2,
        "rougeL": avg_rougeL,
        "em": em_score,
        "samples":n
    }

print("\n=== Q&A Generation Metrics ===")
qa_metrics = compute_qa_metrics(model, tokenizer, val_dataset, DEVICE, max_samples=200)
print(f"Evaluated on {qa_metrics['Samples']} validation examples")
for k, v in qa_metrics.items():
    if k != 'Samples':
        print(f"{k}: {v:.4f}")


In [ ]:
# Show a few example predictions vs references
print("\n--- Example outputs ---")
for i in range(5):
    sample = val_dataset[i]
    question = sample["question"]
    context = sample["context"]
    ref = sample["answer_raw"]
    prompt = f"<|question|> {question} <|context|> {context} <|answer|>"
    gen = generate_text(model, tokenizer, prompt, max_new_tokens=30, temperature=0.0, device=DEVICE)
    pred = gen[gen.find("<|answer|>")+len("<|answer|>"):].strip()
    print(f"\nQ: {question}")
    print(f"Ref: {ref}")
    print(f"Gen: {pred}")